In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Upload dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0])
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['Tanggal'].value_counts()

In [ ]:
def parse_tanggal_with_year(row):
    tanggal_val = row['Tanggal']
    tahun = str(row['Tahun'])

    # Handle NaN or non-string values directly
    if pd.isna(tanggal_val) or not isinstance(tanggal_val, str):
        return pd.NaT

    tanggal_str = tanggal_val.strip()

    # Attempt to parse the date string as is. If it has a year (e.g., '6-Oct-16'), it should work.
    # If it's just '28-Sep', it might raise a ValueError or default to a wrong year.
    try:
        parsed_date = pd.to_datetime(tanggal_str, dayfirst=True)
        # If the original string had a year and it parsed correctly, use it.
        # This check is to avoid cases where '28-Sep' might default to the current year.
        # We assume if the original string did not include a year, it needs the 'Tahun' column.
        # A simpler approach is to always try with year first, and if it fails, try without.
        # But the problem description implies 'missing year' needs 'Tahun'.
        # So, if original parsing results in a valid date, we'll check if the original string
        # actually contained a year. However, this is hard to do without the original string.
        # The safest approach is to assume if parsing fails to resolve a year, use `Tahun`.

        # A robust way: if the string explicitly contains 4 digits, assume it has a year.
        # Otherwise, assume it needs the year from 'Tahun'.
        # Let's refine this to directly address the user's intent: use 'Tahun' for entries missing a year.

        # Try to parse assuming format 'DD-Mon-YYYY' or 'DD-Mon-YY'
        # If it succeeds, it means the year was likely present in the string
        return parsed_date
    except ValueError:
        # If parsing fails, it's likely missing a year (e.g., '28-Sep')
        # Construct a new string with the year from 'Tahun'
        try:
            full_date_str = f"{tanggal_str}-{tahun}"
            return pd.to_datetime(full_date_str, dayfirst=True)
        except ValueError:
            # If even with the year appended it fails, it's an unparseable date
            return pd.NaT

# Apply the custom parsing function
df['Tanggal'] = df.apply(parse_tanggal_with_year, axis=1)

# Cek hasil konversi
df['Tanggal'].isnull().sum()

In [ ]:
df.sample(5)

In [ ]:
df['Qty'] = pd.to_numeric(df['Qty'], errors='coerce')

In [ ]:
df['Qty'] = df['Qty'].fillna(0)

In [ ]:
df['Bulan'] = df['Tanggal'].dt.to_period('M')

In [ ]:
df['Bulan'] = df['Tanggal'].dt.strftime('%Y-%m')

In [ ]:
df['Quarter'] = df['Tanggal'].dt.to_period('Q')

In [ ]:
df['Quarter'] = 'Q' + df['Tanggal'].dt.quarter.astype(str)

In [ ]:
df_clean = df.dropna(subset=['Total IDR']).copy()

df_clean.info()

In [ ]:
df_clean['Total IDR'] = df_clean['Total IDR'].str.replace(',', '').astype(float)

df_clean['Revenue_per_Qty'] = np.where(
    df_clean['Qty'] > 0,
    df_clean['Total IDR'] / df_clean['Qty'],
    np.nan
)

In [ ]:
df_clean.describe()

In [ ]:
df_clean.sort_values('Total IDR', ascending=False).head(10)

In [ ]:
df_clean.sample(5)

#Agregasi Revenue

In [ ]:
# Revenue per Tahun
revenue_tahun = (
    df_clean
    .groupby('Tahun')['Total IDR']
    .sum()
    .reset_index()
    .sort_values('Tahun')
)

print("Revenue per Tahun")
display(revenue_tahun)


# Revenue per Bulan
revenue_bulanan = (
    df_clean
    .groupby(pd.Grouper(key='Tanggal', freq='M'))['Total IDR']
    .sum()
    .reset_index()
    .sort_values('Tanggal')
)

print("Revenue per Bulan")
display(revenue_bulanan)


# Revenue per Quarter
revenue_quarter = (
    df_clean
    .groupby(pd.Grouper(key='Tanggal', freq='Q'))['Total IDR']
    .sum()
    .reset_index()
    .sort_values('Tanggal')
)

print("Revenue per Quarter")
display(revenue_quarter)

##Line Chart

In [ ]:
import matplotlib.ticker as mticker

# --- Line Chart Revenue Bulanan ---
plt.figure(figsize=(10, 6))
plt.plot(revenue_bulanan['Tanggal'], revenue_bulanan['Total IDR'], marker='o')
plt.title('Revenue Bulanan', fontsize=16)
plt.xlabel('Tanggal', fontsize=12)
plt.ylabel('Total Revenue (IDR)', fontsize=12)

# Format y-axis to display full IDR values with comma separators
formatter = mticker.FuncFormatter(lambda x, p: format(int(x), ',d'))
plt.gca().yaxis.set_major_formatter(formatter)

plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('revenue_bulanan.png')
plt.show()

In [ ]:
import matplotlib.ticker as mticker

# --- Line Chart Revenue Tahunan ---
plt.figure(figsize=(10, 6))
plt.plot(revenue_tahun['Tahun'], revenue_tahun['Total IDR'], marker='o')
plt.title('Revenue Tahunan', fontsize=16)
plt.xlabel('Tahun', fontsize=12)
plt.ylabel('Total Revenue (IDR)', fontsize=12)

# Format y-axis to display full IDR values with comma separators
formatter = mticker.FuncFormatter(lambda x, p: format(int(x), ',d'))
plt.gca().yaxis.set_major_formatter(formatter)

# Set x-axis ticks to show all years from 2013-2017
plt.xticks(revenue_tahun['Tahun'].astype(int))

plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('revenue_tahunan.png')
plt.show()

In [ ]:
import matplotlib.ticker as mticker

# --- Line Chart Revenue Quarter ---
plt.figure(figsize=(10, 6))
plt.plot(revenue_quarter['Tanggal'], revenue_quarter['Total IDR'], marker='o')
plt.title('Revenue per Quarter', fontsize=16)
plt.xlabel('Quarter', fontsize=12)
plt.ylabel('Total Revenue (IDR)', fontsize=12)

# Format y-axis to display full IDR values with comma separators
formatter = mticker.FuncFormatter(lambda x, p: format(int(x), ',d'))
plt.gca().yaxis.set_major_formatter(formatter)

plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('revenue_quarter.png')
plt.show()

In [ ]:
revenue_tahun = (
    df_clean
    .groupby('Tahun')['Total IDR']
    .sum()
    .reset_index()
    .sort_values('Tahun')
)

# ==============================
# 3. HITUNG YoY GROWTH (%)
# ==============================

revenue_tahun['YoY_Growth_%'] = revenue_tahun['Total IDR'].pct_change() * 100

display(revenue_tahun)

# ==============================
# 4. VISUALISASI YoY
# ==============================

plt.figure()
plt.plot(revenue_tahun['Tahun'], revenue_tahun['YoY_Growth_%'])
plt.title('Year over Year (YoY) Growth %')
plt.xlabel('Tahun')
plt.ylabel('YoY Growth (%)')
plt.tight_layout()
plt.savefig('revenue_yoy.png')
plt.show()

In [ ]:
df_clean.info()

#RFM Analysis

In [ ]:
# ==============================
# 1. DATA PREPARATION
# ==============================

df_seg = df_clean.dropna(subset=['Perusahaan', 'Tanggal', 'Total IDR']).copy()

snapshot_date = df_seg['Tanggal'].max()

In [ ]:
# ==============================
# 2. RFM CALCULATION
# ==============================

rfm = df_seg.groupby('Perusahaan').agg({
    'Tanggal': lambda x: (snapshot_date - x.max()).days,   # Recency (days)
    'Perusahaan': 'count',                                  # Frequency
    'Total IDR': 'sum'                                       # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']


In [ ]:
# ==============================
# 3. RFM SCORING (1–4 QUARTILE)
# ==============================

rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

In [ ]:
# ==============================
# 4. STRATEGIC SEGMENT LOGIC
# ==============================

def strategic_segment(row):
    if row['R_Score'] == 4 and row['F_Score'] == 4 and row['M_Score'] == 4:
        return "Champions"
    elif row['F_Score'] >= 3 and row['M_Score'] >= 4:
        return "Loyal High Value"
    elif row['M_Score'] == 4 and row['R_Score'] <= 2:
        return "At Risk Big Spender"
    elif row['R_Score'] == 1:
        return "Churn Risk"
    elif row['F_Score'] == 1 and row['M_Score'] == 1:
        return "Low Value"
    else:
        return "Potential Growth"

rfm['Segment'] = rfm.apply(strategic_segment, axis=1)

In [ ]:
# ==============================
# 5. SEGMENT SUMMARY
# ==============================

segment_summary = (
    rfm.groupby('Segment')
       .agg(
           Customer_Count=('Segment', 'count'),
           Total_Revenue=('Monetary', 'sum')
       )
       .reset_index()
)

total_revenue = segment_summary['Total_Revenue'].sum()

segment_summary['Revenue_%'] = (
    segment_summary['Total_Revenue'] / total_revenue * 100
)

segment_summary = segment_summary.sort_values(
    'Total_Revenue', ascending=False
)

display(segment_summary)

In [ ]:
# ==============================
# 6. VISUALISASI JUMLAH CUSTOMER
# ==============================

# Define the custom order for segments
segment_order = [
    "Champions",
    "Loyal High Value",
    "Potential Growth",
    "At Risk Big Spender",
    "Churn Risk",
    "Low Value"
]

# Convert 'Segment' column to a categorical type with the specified order
segment_summary['Segment'] = pd.Categorical(
    segment_summary['Segment'],
    categories=segment_order,
    ordered=True
)

# Sort the DataFrame by the new categorical 'Segment' column
segment_summary_sorted = segment_summary.sort_values('Segment')

plt.figure(figsize=(10, 6))
segment_summary_sorted.plot(x='Segment', y='Customer_Count', kind='bar', legend=False)
plt.title("Customer Segment Distribution", fontsize=16)
plt.xlabel("Segment", fontsize=12)
plt.ylabel("Number of Customers", fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotate and align for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('customer_segment.png')
plt.show()

In [ ]:
import matplotlib.ticker as mticker

# ==============================
# 7. VISUALISASI TOTAL REVENUE PER SEGMENT
# ==============================

plt.figure(figsize=(10, 6))
segment_summary_sorted.plot(x='Segment', y='Total_Revenue', kind='bar', legend=False, color='skyblue')
plt.title("Total Revenue per Customer Segment", fontsize=16)
plt.xlabel("Segment", fontsize=12)
plt.ylabel("Total Revenue (IDR)", fontsize=12)

# Format y-axis to display full IDR values with comma separators
formatter = mticker.FuncFormatter(lambda x, p: format(int(x), ',d'))
plt.gca().yaxis.set_major_formatter(formatter)

plt.xticks(rotation=45, ha='right') # Rotate and align for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('revenue_segment.png')
plt.show()

In [ ]:
rfm.head(10)
rfm.to_csv('rfm_analysis.csv', index=True)